# XGBoost direct
Эксперимент на GPU. Обе ветки используют одинаковые строки с доступным profile28, признаки FE и недельные границы. Цель y / 100000. Для residual отсечение снизу применяется после прибавления профиля.


In [1]:
from pathlib import Path
import json
import gc
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display
DATA = Path('data')
TRAIN_FILE = 'train_solo_track.parquet'
TEST_FILE = 'test_solo_track.parquet'
TRAIN_PATH, TEST_PATH = (DATA / TRAIN_FILE, DATA / TEST_FILE)
if not (TRAIN_PATH.is_file() and TEST_PATH.is_file()) and Path('/kaggle/input').exists():
    train_matches = list(Path('/kaggle/input').rglob(TRAIN_FILE))
    test_matches = list(Path('/kaggle/input').rglob(TEST_FILE))
    if len(train_matches) != 1 or len(test_matches) != 1:
        raise ValueError(f'Укажите TRAIN_PATH и TEST_PATH вручную: train={train_matches}, test={test_matches}')
    TRAIN_PATH, TEST_PATH = (train_matches[0], test_matches[0])
OUT = Path('artifacts/xgb_direct/features')
MODEL_DIR = OUT.parent / 'models'
IMPORTANCE_DIR = OUT.parent / 'importance'
POSTPROCESS_DIR = OUT.parent / 'postprocessing'
for directory in (OUT, MODEL_DIR, IMPORTANCE_DIR, POSTPROCESS_DIR):
    directory.mkdir(parents=True, exist_ok=True)
TARGET = 'target_1h'
STATUSES = [f'status_{i}' for i in range(1, 7)]
STEP = pd.Timedelta(minutes=30)
TARGET_SCALE = 100000
MAX_TREES = 4000
EARLY_STOPPING_ROUNDS = 150
WINDOW_OFFSETS = tuple(range(-4, 5))
LAG_MAX_LOOKBACK_HOURS = 24
LAG_MIN_PAIRS = 336
LAG_MIN_CORRELATION = 0.05
PERMUTATION_ENABLED = True
PERMUTATION_TOP_N = 10
PERMUTATION_REPEATS = 3
PERMUTATION_MAX_DAYS = 7
FEATURE_PIPELINE_VERSION = '1030_windows_v4'
RESIDUAL = False


## Данные и временные границы
Проверяется полная получасовая панель всех маршрутов. Границы задаются до расчёта любых статистик, использующих цель. Словарь корреляционных лагов обучается только до границы train.


In [2]:
history = pd.read_parquet(TRAIN_PATH, columns=['route_id', 'timestamp', TARGET, *STATUSES]).sort_values(['timestamp', 'route_id'])
test = pd.read_parquet(TEST_PATH, columns=['id', 'route_id', 'timestamp'])
times = pd.DatetimeIndex(history.timestamp.unique())
routes = np.sort(history.route_id.unique())
if not (history.notna().all().all() and test.notna().all().all()):
    raise ValueError('В данных есть пропуски')
if not not history.duplicated(['route_id', 'timestamp']).any():
    raise ValueError('Повторяется ключ маршрута и времени')
if not times.equals(pd.date_range(times[0], times[-1], freq=STEP)):
    raise ValueError('Нарушена получасовая сетка')
if not len(history) == len(times) * len(routes):
    raise ValueError('Неполная панель маршрутов')
if not (test.id.is_unique and (not test.duplicated(['route_id', 'timestamp']).any())):
    raise ValueError('Повторы идентификаторов или ключей теста')
end = times[-1]
if not (end.hour, end.minute) == (10, 30):
    raise ValueError('Последняя точка истории должна быть в 10:30')
expected = pd.MultiIndex.from_product([routes, [end + h * STEP for h in range(1, 9)]], names=['route_id', 'timestamp'])
if not expected.difference(pd.MultiIndex.from_frame(test[['route_id', 'timestamp']])).empty:
    raise ValueError('Тест не покрывает все ожидаемые маршруты и горизонты')
if not len(test) == len(expected):
    raise ValueError('Число строк теста отличается от ожидаемого')
values = {name: history[name].to_numpy(dtype=np.float32).reshape(len(times), len(routes)) for name in [TARGET, *STATUSES]}
del history
if not all((np.isfinite(array).all() for array in values.values())):
    raise ValueError('В числовых рядах есть бесконечные значения')
dates = times[(times.hour == 10) & (times.minute == 30) & (times + 8 * STEP <= end)]
if len(dates) <= 28:
    raise ValueError('Недостаточно дат для временного разбиения')
if len(dates) < 63:
    raise ValueError('Недостаточно истории для трёх недель подбора')
holdout_start = dates[-14]
folds = []
for start in (len(dates)-35, len(dates)-28, len(dates)-21):
    folds.append(dict(train=dates[:start-7], val=dates[start-7:start], tuning=dates[start:start+7], test=dates[-14:]))
split_dates = folds[-1]
tune_start = split_dates['tuning'][0]
LAG_VALIDATION_FIT_BEFORE = split_dates['val'][0].normalize()
display(pd.DataFrame([dict(fold=i+1, period=k, first=v[0], last=v[-1], days=len(v)) for i,f in enumerate(folds) for k,v in f.items()]))


,fold,period,first,last,days
0,1,train,2025-07-28 10:30:00,2025-09-19 10:30:00,54
1,1,val,2025-09-20 10:30:00,2025-09-26 10:30:00,7
2,1,tuning,2025-09-27 10:30:00,2025-10-03 10:30:00,7
3,1,test,2025-10-18 10:30:00,2025-10-31 10:30:00,14
4,2,train,2025-07-28 10:30:00,2025-09-26 10:30:00,61
5,2,val,2025-09-27 10:30:00,2025-10-03 10:30:00,7
6,2,tuning,2025-10-04 10:30:00,2025-10-10 10:30:00,7
7,2,test,2025-10-18 10:30:00,2025-10-31 10:30:00,14
8,3,train,2025-07-28 10:30:00,2025-10-03 10:30:00,68
9,3,val,2025-10-04 10:30:00,2025-10-10 10:30:00,7


## Исторические признаки и окна
Сохраняются признаки присланного ноутбука: уровни цели и статусов, средние за 2/6 часов, изменения за час, статусные суммы и отношения, суточные лаги, отклонение от среднего и отношение текущего уровня к предыдущим дням.

Вокруг каждого лага строится окно ±4 получасовых шага. Для последних значений и сглаженных сигналов доступны только точки не позже собственного старта: положительные сдвиги не создаются. Одинаковые источники не дублируются: например, соседние окна target_back_0/1/2/4 объединяются в target_back_0…8. Для status_latest добавляются back_1…4. Лаги сутки/двое назад и сезонные слоты получают все девять соседей. Исходные центральные признаки сохранены. Эквивалентные change_1h статусов 4–6 заменены на diff_hourly.

Суммы статусов не являются числом уникальных товаров. Названия warehouse_total_load, hub_buffer и hub_gate_bottleneck обозначают формулы, а не подтверждённые физические показатели. Недоступная история остаётся NaN.

Для цели и каждого статуса рассчитываются общая сумма по всем маршрутам в доступный момент и её изменение за час. Соседние исторические значения используют только левую часть окна у старта.


In [3]:
import pyarrow as pa
import pyarrow.parquet as pq
target = values[TARGET]

def past(array, positions):
    positions = np.asarray(positions, dtype=np.int64)
    valid = (positions >= 0) & (positions < len(array))
    result = array[np.clip(positions, 0, len(array) - 1)].astype(np.float32)
    result[~valid] = np.nan
    return result

def average_available(matrices):
    total = np.zeros_like(matrices[0], dtype=np.float32)
    count = np.zeros_like(total)
    for matrix in matrices:
        valid = np.isfinite(matrix)
        total += np.where(valid, matrix, 0)
        count += valid
    return np.divide(total, count, out=np.full_like(total, np.nan), where=count > 0)

def rolling(array, steps):
    return pd.DataFrame(array).rolling(steps, min_periods=1).mean().to_numpy(dtype=np.float32)
means = {(name, steps): rolling(array, steps) for name, array in values.items() for steps in (4, 12)}
level24 = rolling(target, 48)
positions = np.arange(len(times))
extra_signals = {}
for name in STATUSES[:3]:
    extra_signals[name + '_hourly'] = values[name] + past(values[name], positions - 1)
extra_signals['warehouse_total_load'] = sum((extra_signals[name + '_hourly'] for name in STATUSES[:3]))
extra_signals['warehouse_processing_rate'] = extra_signals['status_3_hourly'] / (extra_signals['status_1_hourly'] + np.float32(1e-05))
for name in STATUSES[3:]:
    extra_signals[name + '_diff_hourly'] = values[name] - past(values[name], positions - 2)
extra_signals['hub_buffer'] = values['status_5'] - values['status_4']
extra_signals['hub_gate_bottleneck'] = extra_signals['status_4_diff_hourly'] - extra_signals['status_6_diff_hourly']
extra_signals['target_acceleration'] = past(target, positions - 1) - past(target, positions - 2)
hub_signal = extra_signals['status_6_diff_hourly']
extra_signals['target_vs_mean_6h'] = target - means[TARGET, 12]
recent = means[TARGET, 4]
usual = average_available([past(recent, positions - day * 48) for day in range(1, 8)])
extra_signals['morning_level_ratio'] = np.divide(recent, usual, out=np.full_like(recent, np.nan), where=usual > 0)

def offset_name(base, offset):
    if offset == 0:
        return base
    return f"{base}_offset_{('m' if offset < 0 else 'p')}{abs(offset)}"

def add_window(common, base, array, anchor, offsets=WINDOW_OFFSETS):
    for offset in offsets:
        common[offset_name(base, offset)] = past(array, anchor + offset)

def common_features(origins):
    common = {}
    local_target_lags = sorted({k - offset for k in (0, 1, 2, 4) for offset in WINDOW_OFFSETS if k - offset >= 0})
    for lag in local_target_lags:
        common[f'target_back_{lag}'] = past(target, origins - lag)
    causal_offsets = tuple((offset for offset in WINDOW_OFFSETS if offset <= 0))
    for name, array in values.items():
        for steps in (4, 12):
            add_window(common, f'{name}_mean_{steps}', means[name, steps], origins, causal_offsets)
        if name in STATUSES:
            common[f'{name}_latest'] = array[origins]
            for lag in range(1, 5):
                common[f'{name}_back_{lag}'] = past(array, origins - lag)
            if name in STATUSES[:3]:
                change = array - past(array, positions - 2)
                add_window(common, f'{name}_change_1h', change, origins, causal_offsets)
    for name, array in extra_signals.items():
        add_window(common, name, array, origins, causal_offsets)
    for lag in (48, 96):
        add_window(common, f'target_1h_lag_{lag}', target, origins - lag)
    add_window(common, 'warehouse_total_load_lag_48', extra_signals['warehouse_total_load'], origins - 48)
    add_window(common, 'status_6_diff_hourly_lag_48', hub_signal, origins - 48)
    return common
CORRELATION_COLUMNS = ['selected_status6_lag_steps', 'selected_status6_correlation', 'selected_status6_pair_count', 'status6_lag_valid', 'lagged_status6_signal', 'lagged_status6_signal_available']

for name, series in values.items():
    network = series.sum(axis=1, dtype=np.float64).astype(np.float32)[:, None]
    for suffix, signal in [('latest', network), ('change_1h', network - past(network, positions-2))]:
        extra_signals[f'network_{name}_{suffix}'] = np.broadcast_to(signal, target.shape)


## Корреляционный лаг маршрута
На полной получасовой истории train ищется максимальная положительная Pearson-корреляция target(u) с diff_status6(u−L), L=2…48 шагов. Минимум 336 пар и корреляция >0,05 — эвристики, не тест значимости.

Выбранный L фиксирован для маршрута. Для горизонта h нужен сигнал origin+h−L. Помимо центра добавляются соседи ±2 часа, каждый доступен только при собственном источнике ≤origin. У каждого соседа есть флаг доступности. При слабой связи лаг и сигнал NaN. Для финального refit словарь пересчитывается на всей истории; оценочные parquet и модели сохраняются отдельно.

Для выбора лага и его применения из цели и изменения статуса 6 вычитается среднее того же получаса за предыдущие семь суток. Для каждого окна словарь строится заново до начала val.
Исходный лаг статуса 6 и лаг по остаткам сохраняются отдельными группами.


In [4]:
def calculate_dynamic_shoulder(target_values, status6_diff, timestamps, route_ids, *, fit_before, max_lookback_hours=24, min_pairs=336, min_correlation=0.05, route_block_size=128):
    grid = pd.DatetimeIndex(timestamps)
    route_ids = np.asarray(route_ids)
    target_values, status6_diff = (np.asarray(target_values), np.asarray(status6_diff))
    if target_values.shape != status6_diff.shape or target_values.shape != (len(grid), len(route_ids)):
        raise ValueError('Ожидаются два массива [время, маршрут] одной формы')
    if len(grid) == 0 or not grid.equals(pd.date_range(grid[0], periods=len(grid), freq=STEP)):
        raise ValueError('Для выбора лага нужна упорядоченная получасовая сетка без дубликатов')
    if len(np.unique(route_ids)) != len(route_ids):
        raise ValueError('Идентификаторы маршрутов повторяются')
    steps = float(max_lookback_hours) * 2
    if not np.isfinite(steps) or steps < 2 or (not steps.is_integer()):
        raise ValueError('Граница поиска должна задавать целое число получасов, не меньше двух')
    if not isinstance(min_pairs, (int, np.integer)) or min_pairs < 3:
        raise ValueError('min_pairs должен быть целым числом >=3')
    if not 0 <= min_correlation <= 1 or route_block_size < 1:
        raise ValueError('Некорректные параметры выбора корреляции')
    max_lag = int(steps)
    stop = int(grid.searchsorted(pd.Timestamp(fit_before), side='left'))
    anchors = np.arange(max_lag + 2, stop)
    best_corr = np.full(len(route_ids), np.nan)
    best_lag = np.full(len(route_ids), np.nan)
    best_pairs = np.zeros(len(route_ids), dtype=np.int32)
    for start in range(0, len(route_ids), route_block_size):
        finish = min(start + route_block_size, len(route_ids))
        x = target_values[anchors, start:finish].astype(np.float64)
        for lag in range(2, max_lag + 1):
            z = status6_diff[anchors - lag, start:finish].astype(np.float64)
            valid = np.isfinite(x) & np.isfinite(z)
            count = valid.sum(axis=0)
            xx, zz = (np.where(valid, x, 0), np.where(valid, z, 0))
            divisor = np.maximum(count, 1)
            sx, sz = (xx.sum(axis=0), zz.sum(axis=0))
            covariance = (xx * zz).sum(axis=0) - sx * sz / divisor
            vx = np.maximum((xx * xx).sum(axis=0) - sx * sx / divisor, 0)
            vz = np.maximum((zz * zz).sum(axis=0) - sz * sz / divisor, 0)
            denominator = np.sqrt(vx * vz)
            corr = np.divide(covariance, denominator, out=np.full(finish - start, np.nan), where=(count >= min_pairs) & (denominator > 0))
            corr = np.clip(corr, -1, 1)
            previous = best_corr[start:finish]
            better = np.isfinite(corr) & (~np.isfinite(previous) | (corr > previous))
            best_corr[start:finish][better] = corr[better]
            best_lag[start:finish][better] = lag
            best_pairs[start:finish][better] = count[better]
    accepted = np.isfinite(best_corr) & (best_corr > min_correlation)
    return pd.DataFrame({'route_id': route_ids, 'selected_status6_lag_steps': np.where(accepted, best_lag, np.nan), 'selected_status6_correlation': best_corr, 'selected_status6_pair_count': best_pairs, 'status6_lag_valid': accepted.astype(np.int8)})

def correlation_block(frame, h, lag_table, signal_array, prefix, chunk_rows=200000):
    if h not in range(1, 9) or chunk_rows < 1:
        raise ValueError('Некорректный горизонт или размер блока')
    by_route = lag_table.set_index('route_id').reindex(routes)
    route_index = pd.Index(routes)
    lag = by_route.selected_status6_lag_steps.to_numpy(dtype=np.float32)
    extra_columns = [offset_name(base, offset) for offset in WINDOW_OFFSETS if offset != 0 for base in ('lagged_status6_signal', 'lagged_status6_signal_available')]
    names = CORRELATION_COLUMNS + extra_columns
    result = np.full((len(frame), len(names)), np.nan, dtype=np.float32)
    for start in range(0, len(frame), chunk_rows):
        finish = min(start + chunk_rows, len(frame))
        block = frame.iloc[start:finish]
        origin = times.get_indexer(pd.DatetimeIndex(block.cutoff))
        route = route_index.get_indexer(block.route_id)
        if (origin < 0).any() or (route < 0).any():
            raise ValueError('Строка не сопоставлена с историей')
        selected = lag[route]
        for j, name in enumerate(CORRELATION_COLUMNS[:4]):
            result[start:finish, j] = by_route[name].to_numpy(dtype=np.float32)[route]
        for offset in WINDOW_OFFSETS:
            source = origin + h - np.nan_to_num(selected, nan=0).astype(np.int64) + offset
            available = np.isfinite(selected) & (source >= 0) & (source <= origin) & (source < len(times))
            signal = signal_array[np.clip(source, 0, len(times) - 1), route].astype(np.float32)
            available &= np.isfinite(signal)
            result[start:finish, names.index(offset_name('lagged_status6_signal', offset))] = np.where(available, signal, np.nan)
            result[start:finish, names.index(offset_name('lagged_status6_signal_available', offset))] = available.astype(np.float32)
    names = [prefix + name for name in names]
    frame = frame.drop(columns=names, errors='ignore')
    return pd.concat([frame, pd.DataFrame(result, columns=names, index=frame.index)], axis=1)
def remove_seasonality(series):
    typical = average_available([past(series, positions - 48*d) for d in range(1,8)])
    return series - typical

residual_target = remove_seasonality(target)
residual_signal = remove_seasonality(hub_signal)

def fit_lags(boundary):
    tables = []
    for y, signal, prefix in ((target,hub_signal,''),(residual_target,residual_signal,'residual_')):
        table = calculate_dynamic_shoulder(y,signal,times,routes,fit_before=boundary,max_lookback_hours=LAG_MAX_LOOKBACK_HOURS,min_pairs=LAG_MIN_PAIRS,min_correlation=LAG_MIN_CORRELATION)
        tables.append(table.rename(columns={c:prefix+c for c in table.columns if c != 'route_id'}))
    return tables[0].merge(tables[1],on='route_id',validate='one_to_one')

def add_correlation_features(frame,h,lag_table):
    frame = correlation_block(frame,h,lag_table,hub_signal,'')
    residual_table = lag_table[['route_id',*['residual_'+c for c in CORRELATION_COLUMNS[:4]]]].copy()
    residual_table.columns = ['route_id',*CORRELATION_COLUMNS[:4]]
    return correlation_block(frame,h,residual_table,residual_signal,'residual_')

validation_lag_table = fit_lags(LAG_VALIDATION_FIT_BEFORE)
validation_lag_table.to_csv(OUT.parent / 'status6_lags_validation.csv', index=False)


## Матрицы восьми горизонтов
Каждая историческая строка — маршрут и старт в 10:30 с известной меткой данного горизонта. Сезонные значения берутся относительно прогнозируемого слота, включая ±2 часа. Средние slot_daily7/slot_weekly4 получают отдельное значение для каждого сдвига, а не усреднение внутри окна. Центральные profile28 и overlap используются только постобработкой. Календарь относится к будущему слоту.

Для профилей slot_daily7 и slot_weekly4 вычисляются сдвиг пика в минутах, размах и разность средних правой и левой частей окна. При нескольких равных максимумах используется их средняя позиция. Для календаря остаются день недели, выходной и циклические координаты дня недели.


In [5]:
def horizon_dataset(h, origins):
    slot = origins + h
    dates = times[origins] + h * STEP
    common = common_features(origins)
    frame = pd.DataFrame({'cutoff': times[origins].repeat(len(routes)), 'timestamp': dates.repeat(len(routes)), 'route_id': np.tile(routes, len(origins)), **{name: matrix.ravel() for name, matrix in common.items()}})
    calendar = dict(dayofweek=dates.dayofweek, is_weekend=(dates.dayofweek >= 5).astype(np.int8))
    calendar['weekday_sin'] = np.sin(2 * np.pi * dates.dayofweek / 7)
    calendar['weekday_cos'] = np.cos(2 * np.pi * dates.dayofweek / 7)
    for name, array in calendar.items():
        frame[name] = np.repeat(array, len(routes))
    frame = add_correlation_features(frame, h, validation_lag_table)
    seasonal_columns = {}
    for offset in WINDOW_OFFSETS:
        for day in (1, 2, 7):
            seasonal_columns[offset_name(f'slot_days_back_{day}', offset)] = past(target, slot - day * 48 + offset).ravel()
        for name, days in [('slot_daily7', range(1, 8)), ('slot_weekly4', (7, 14, 21, 28))]:
            seasonal_columns[offset_name(name, offset)] = average_available([past(target, slot - day * 48 + offset) for day in days]).ravel()
    for base in ('slot_daily7','slot_weekly4'):
        window = np.column_stack([seasonal_columns[offset_name(base,k)] for k in WINDOW_OFFSETS])
        valid = np.isfinite(window)
        maximum = np.where(valid, window, -np.inf).max(axis=1)
        minimum = np.where(valid, window, np.inf).min(axis=1)
        peak = valid & (window == maximum[:,None])
        count = peak.sum(axis=1)
        shift = np.divide(peak @ np.asarray(WINDOW_OFFSETS, dtype=float), count, out=np.full(len(window),np.nan), where=count>0)
        seasonal_columns[base+'_peak_minutes'] = shift * 30
        seasonal_columns[base+'_range'] = np.where(count>0, maximum-minimum, np.nan)
        seasonal_columns[base+'_right_minus_left'] = window[:,5:].mean(axis=1)-window[:,:4].mean(axis=1)
    frame = pd.concat([frame, pd.DataFrame(seasonal_columns, index=frame.index)], axis=1)
    profile = np.zeros((len(origins), len(routes)))
    counts = np.zeros_like(profile)
    for day in range(1, 29):
        level = past(level24, origins - day * 48)
        adjusted = past(target, slot - day * 48) * level24[origins] / np.maximum(level, 1)
        profile += np.nan_to_num(adjusted)
        counts += np.isfinite(adjusted)
    frame['profile28'] = np.divide(profile, counts, out=np.full_like(profile, np.nan), where=counts > 0).ravel()
    if h == 1:
        lower = np.zeros_like(target[origins])
        upper = past(target, origins - 2)
        for position in (origins - 1, origins):
            total = past(target, position)
            lower, upper = (np.maximum(0, total - upper), np.minimum(total, total - lower))
        seasonal = frame[['slot_daily7', 'slot_weekly4']].mean(axis=1).to_numpy()
        frame['overlap'] = np.where((lower <= upper).ravel(), (lower + upper).ravel() / 2 + seasonal / 2, seasonal)
    else:
        frame['overlap'] = np.nan
    raw_target_sources = []
    for lag in range(9):
        raw_target_sources.append((f'target_back_{lag}', -lag, lag in (0, 1, 2, 4)))
    for lag in (48, 96):
        for offset in WINDOW_OFFSETS:
            raw_target_sources.append((offset_name(f'target_1h_lag_{lag}', offset), -lag + offset, offset == 0))
    for day in (1, 2, 7):
        for offset in WINDOW_OFFSETS:
            raw_target_sources.append((offset_name(f'slot_days_back_{day}', offset), h - day * 48 + offset, offset == 0))
    seen, redundant = (set(), [])
    for name, source, central in sorted(raw_target_sources, key=lambda item: not item[2]):
        if source in seen:
            redundant.append(name)
        else:
            seen.add(source)
    frame = frame.drop(columns=redundant)
    numeric = frame.columns.difference(['cutoff', 'timestamp', 'route_id'])
    return frame.astype({name: 'float32' for name in numeric}).copy()

def write_horizon(h):
    available_origins = np.flatnonzero((times.hour == 10) & (times.minute == 30) & (np.arange(len(times)) + h < len(times)))
    path = OUT / f'history_h{h}.parquet'
    frame = horizon_dataset(h, available_origins)
    frame['y'] = target[available_origins + h].ravel()
    if not frame.columns.is_unique or not ((frame.cutoff.dt.hour == 10) & (frame.cutoff.dt.minute == 30)).all():
        raise ValueError('Нарушена схема признаков или время старта')
    frame.to_parquet(path, index=False, compression='zstd')
    features = horizon_dataset(h, np.array([len(times) - 1]))
    requests = test.loc[test.timestamp.eq(end + h * STEP)]
    future = requests.merge(features, on=['route_id', 'timestamp'], how='left', validate='one_to_one')
    if not future.cutoff.eq(end).all() or not future.id.equals(requests.id.reset_index(drop=True)):
        raise ValueError('Ошибка сопоставления тестовых строк')
    future.to_parquet(OUT / f'submission_h{h}.parquet', index=False)
    print(f'h={h}: история {len(frame):,}, submission {len(future):,}, модельных признаков {len(frame.columns) - 6}')
for h in range(1, 9):
    write_horizon(h)
feature_names = list(dict.fromkeys((name for h in range(1, 9) for name in pq.read_schema(OUT / f'history_h{h}.parquet').names)))
FEATURE_GROUPS = {'calendar': ['dayofweek', 'is_weekend', 'weekday_sin', 'weekday_cos'], 'local_statuses': [name for name in feature_names if any((name.startswith(prefix) for prefix in ('status_1_hourly', 'status_2_hourly', 'status_3_hourly', 'warehouse_total_load', 'warehouse_processing_rate'))) and 'lag_48' not in name], 'hub_combinations': [name for name in feature_names if any((name.startswith(prefix) for prefix in ('status_4_diff_hourly', 'status_5_diff_hourly', 'status_6_diff_hourly', 'hub_buffer', 'hub_gate_bottleneck'))) and 'lag_48' not in name], 'historical_lags': [name for name in feature_names if 'lag_48' in name or 'lag_96' in name], 'target_change': [name for name in feature_names if name.startswith('target_acceleration')], 'correlation_lag': [name for name in feature_names if name in CORRELATION_COLUMNS or name.startswith('lagged_status6_signal')], 'seasonal_windows': [name for name in feature_names if name.startswith('slot_')], 'recent_target': [name for name in feature_names if name.startswith('target_back_')], 'recent_statuses': [name for name in feature_names if any((name.startswith(f'status_{i}_back_') or name == f'status_{i}_latest' for i in range(1, 7)))]}

del values, means, level24, extra_signals, positions, recent, usual
FEATURE_GROUPS['network'] = [name for name in feature_names if name.startswith('network_')]
FEATURE_GROUPS['residual_correlation_lag'] = [name for name in feature_names if name.startswith('residual_')]
(OUT.parent / 'feature_pipeline.json').write_text(json.dumps({'version': FEATURE_PIPELINE_VERSION, 'window_offsets_steps': list(WINDOW_OFFSETS), 'training_origin': '10:30', 'validation_lag_fit_before': str(LAG_VALIDATION_FIT_BEFORE), 'splits': {key: [str(period.min()), str(period.max())] for key, period in split_dates.items()}, 'feature_groups': FEATURE_GROUPS}, ensure_ascii=False, indent=2))


C:\Users\artem\AppData\Local\Temp\ipykernel_2288\554008329.py:6: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  calendar = dict(dayofweek=dates.dayofweek, is_weekend=(dates.dayofweek >= 5).astype(np.int8))
C:\Users\artem\AppData\Local\Temp\ipykernel_2288\554008329.py:6: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  calendar = dict(dayofweek=dates.dayofweek, is_weekend=(dates.dayofweek >= 5).astype(np.int8))
C:\Users\artem\AppData\Local\Temp\ipykernel_2288\554008329.py:7: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  calendar['weekday_sin'] = np.sin(2 * np.pi * dates.dayofweek / 7)
C:\Users\artem\AppData\Local\Temp\ipykernel_2288\554008329.py:8: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a

h=1: история 96,000, submission 1,000, модельных признаков 378


C:\Users\artem\AppData\Local\Temp\ipykernel_2288\554008329.py:6: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  calendar = dict(dayofweek=dates.dayofweek, is_weekend=(dates.dayofweek >= 5).astype(np.int8))
C:\Users\artem\AppData\Local\Temp\ipykernel_2288\554008329.py:6: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  calendar = dict(dayofweek=dates.dayofweek, is_weekend=(dates.dayofweek >= 5).astype(np.int8))
C:\Users\artem\AppData\Local\Temp\ipykernel_2288\554008329.py:7: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  calendar['weekday_sin'] = np.sin(2 * np.pi * dates.dayofweek / 7)
C:\Users\artem\AppData\Local\Temp\ipykernel_2288\554008329.py:8: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a

h=2: история 96,000, submission 1,000, модельных признаков 380


C:\Users\artem\AppData\Local\Temp\ipykernel_2288\554008329.py:6: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  calendar = dict(dayofweek=dates.dayofweek, is_weekend=(dates.dayofweek >= 5).astype(np.int8))
C:\Users\artem\AppData\Local\Temp\ipykernel_2288\554008329.py:6: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  calendar = dict(dayofweek=dates.dayofweek, is_weekend=(dates.dayofweek >= 5).astype(np.int8))
C:\Users\artem\AppData\Local\Temp\ipykernel_2288\554008329.py:7: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  calendar['weekday_sin'] = np.sin(2 * np.pi * dates.dayofweek / 7)
C:\Users\artem\AppData\Local\Temp\ipykernel_2288\554008329.py:8: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a

h=3: история 96,000, submission 1,000, модельных признаков 382


... (7 outputs left out)

h=7: история 96,000, submission 1,000, модельных признаков 390


C:\Users\artem\AppData\Local\Temp\ipykernel_2288\554008329.py:6: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  calendar = dict(dayofweek=dates.dayofweek, is_weekend=(dates.dayofweek >= 5).astype(np.int8))
C:\Users\artem\AppData\Local\Temp\ipykernel_2288\554008329.py:6: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  calendar = dict(dayofweek=dates.dayofweek, is_weekend=(dates.dayofweek >= 5).astype(np.int8))
C:\Users\artem\AppData\Local\Temp\ipykernel_2288\554008329.py:7: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  calendar['weekday_sin'] = np.sin(2 * np.pi * dates.dayofweek / 7)
C:\Users\artem\AppData\Local\Temp\ipykernel_2288\554008329.py:8: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a

h=8: история 96,000, submission 1,000, модельных признаков 392


12072

## Три окна и финальный прогноз
Одна конфигурация на горизонт. Валидация перед tuning выбирает число деревьев. Обе ветки используют MAE для early stopping. Прогнозы, метрики и submission сохраняются в artifacts/xgb_direct.


In [6]:
def mean_daily_score(frame, prediction):
    actual = frame.y.to_numpy(dtype=np.float64)
    errors = np.asarray(prediction, dtype=np.float64) - actual
    totals = pd.DataFrame({'cutoff': frame.cutoff.dt.normalize().to_numpy(), 'volume': actual, 'error': errors, 'absolute': np.abs(errors)}).groupby('cutoff').sum()
    denominator = totals.volume.where(totals.volume > 0)
    return ((totals.absolute + totals.error.abs()) / denominator).mean()

def score_row(frame, predictions, split):
    actual = frame.y.to_numpy(dtype=np.float64)
    errors = np.asarray(predictions, dtype=np.float64) - actual
    volume = actual.sum()
    wape = np.abs(errors).sum() / volume if volume > 0 else np.nan
    signed_bias = errors.sum() / volume if volume > 0 else np.nan
    return {'выборка': split, 'даты': frame.cutoff.dt.normalize().nunique(), 'строки': len(frame), 'WAPE': wape, 'Relative Bias': signed_bias, '|Relative Bias|': abs(signed_bias), 'метрика конкурса': wape + abs(signed_bias), 'среднее по дням': mean_daily_score(frame, predictions)}
TARGET_SCALE = 100000

def model_columns(data):
    return data.columns.drop(['timestamp', 'cutoff', 'y', 'profile28', 'overlap']).tolist()


In [7]:
from xgboost import XGBRegressor

def matrix(frame, columns):
    result = frame[columns].copy()
    result['route_id'] = pd.Categorical(result.route_id,categories=routes)
    return result

def offset(frame):
    if RESIDUAL:
        return frame.profile28.to_numpy(dtype=float)
    return np.zeros(len(frame))

def fit_model(frame,columns,trees,early=None):
    options = {}
    kwargs = {}
    if early is not None:
        options['early_stopping_rounds'] = EARLY_STOPPING_ROUNDS
        kwargs['eval_set'] = [(matrix(early,columns),(early.y-offset(early))/TARGET_SCALE)]
    model = XGBRegressor(device='cuda',tree_method='hist',enable_categorical=True,
                         objective='reg:squarederror',eval_metric='mae',max_depth=6,
                         learning_rate=0.05,n_estimators=trees,random_state=42,**options)
    model.fit(matrix(frame,columns),(frame.y-offset(frame))/TARGET_SCALE,verbose=False,**kwargs)
    config = json.loads(model.get_booster().save_config())
    if not config['learner']['generic_param']['device'].startswith('cuda'):
        raise RuntimeError('XGBoost требует GPU с поддержкой CUDA')
    return model

def predict(model, frame, columns):
    return np.maximum(0,offset(frame)+TARGET_SCALE*model.predict(matrix(frame,columns)))

lag_tables = [fit_lags(f['val'][0].normalize()) for f in folds[:-1]]+[validation_lag_table]
full_lags = fit_lags(end+STEP)
parts, reports, submission_parts, importance = [], [], [], []
for h in range(1,9):
    history = pd.read_parquet(OUT/f'history_h{h}.parquet')
    history = history.loc[np.isfinite(history.profile28)].copy()
    columns = model_columns(history)
    for i,(fold,table) in enumerate(zip(folds,lag_tables)):
        frame = add_correlation_features(history,h,table)
        train = frame.loc[frame.cutoff.isin(fold['train'])]
        early = frame.loc[frame.cutoff.isin(fold['val'])]
        model = fit_model(train,columns,MAX_TREES,early)
        trees = model.best_iteration+1
        rows = []
        for split, days in fold.items():
            sample = frame.loc[frame.cutoff.isin(days)]
            pred = predict(model,sample,columns)
            rows.append(score_row(sample,pred,split))
            if split=='tuning' or (i==len(folds)-1 and split=='test'):
                part = sample[['cutoff','route_id','timestamp','y','profile28']].copy()
                part['horizon'],part['period'],part['xgboost'] = h,split,pred
                part['model_info_through'] = fold['val'][-1]+8*STEP
                parts.append(part)
        reports.extend([dict(horizon=h,fold=i+1,trees=trees,**row) for row in rows])
        display(pd.DataFrame(rows))
    frame = add_correlation_features(history,h,full_lags)
    future = add_correlation_features(pd.read_parquet(OUT/f'submission_h{h}.parquet'),h,full_lags)
    model = fit_model(frame,columns,trees)
    model.save_model(MODEL_DIR/f'h{h}.ubj')
    print('Оценка test независима от финального переобучения и получена до него. Даты ранее изучались.')
    display(pd.DataFrame(rows+[score_row(frame,predict(model,frame,columns),'вся история после refit')]))
    importance.append(pd.Series(model.feature_importances_,index=columns,name=h))
    part = future[['id']].copy()
    part['y_pred'] = predict(model,future,columns)
    submission_parts.append(part)
scores = pd.concat(parts,ignore_index=True)
scores.to_parquet(OUT.parent/'xgb_predictions.parquet',index=False)
pd.DataFrame(reports).to_csv(OUT.parent/'metrics.csv',index=False)
for split,part in scores.groupby('period'):
    display(pd.DataFrame([score_row(part,part.xgboost,split),score_row(part,part.profile28,split+' profile28')]))
predictions = pd.concat(submission_parts)
submission = test[['id']].merge(predictions,on='id',how='left',validate='one_to_one')
if len(predictions)!=len(test) or not np.isfinite(submission.y_pred).all():
    raise ValueError('Неполный submission')
submission.to_csv(OUT.parent/'submission.csv',index=False)

tuning = scores.loc[scores.period.eq('tuning')]
control = []
for stop in (1,2):
    boundary = folds[stop]['tuning'][0]
    earlier = tuning.loc[tuning.cutoff < boundary]
    later = tuning.loc[tuning.cutoff.isin(folds[stop]['tuning'])]
    def mix(frame, weight):
        return (1-weight)*frame.xgboost + weight*frame.profile28
    weight = min(np.linspace(0,1,21),key=lambda w:mean_daily_score(earlier,mix(earlier,w)))
    for name,frame in [('train',earlier),('val',later)]:
        for label,w in [('XGBoost',0.),('XGBoost + profile28',weight)]:
            control.append(dict(window=stop,model=label,profile_weight=w,**score_row(frame,mix(frame,w),name)))
control = pd.DataFrame(control)
display(control)
control.to_csv(OUT.parent/'profile_control.csv',index=False)
importance_table = pd.concat(importance,axis=1)
importance_table['mean_all_8'] = importance_table.fillna(0).sum(axis=1)/8
importance_table = importance_table.sort_values('mean_all_8',ascending=False)
importance_table.to_csv(OUT.parent/'feature_importance.csv')
display(importance_table)


c:\Users\artem\projects\experiments\.venv\Lib\site-packages\xgboost\core.py:569: UserWarning: [13:54:30] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


,выборка,даты,строки,WAPE,Relative Bias,|Relative Bias|,метрика конкурса,среднее по дням
0,train,53,53000,0.160777,-0.000038,0.000038,0.160814,0.171482
1,val,7,7000,0.269290,-0.053892,0.053892,0.323182,0.322357
2,tuning,7,7000,0.275077,-0.026479,0.026479,0.301556,0.301797
3,test,14,14000,0.278925,-0.084640,0.084640,0.363565,0.363231


,выборка,даты,строки,WAPE,Relative Bias,|Relative Bias|,метрика конкурса,среднее по дням
0,train,60,60000,0.189536,-0.000077,0.000077,0.189613,0.205089
1,val,7,7000,0.272344,-0.009565,0.009565,0.281909,0.294009
2,tuning,7,7000,0.278987,-0.040686,0.040686,0.319673,0.320568
3,test,14,14000,0.271881,-0.063445,0.063445,0.335325,0.334944


,выборка,даты,строки,WAPE,Relative Bias,|Relative Bias|,метрика конкурса,среднее по дням
0,train,67,67000,0.175008,-0.000045,0.000045,0.175053,0.187592
1,val,7,7000,0.276307,-0.038941,0.038941,0.315248,0.317125
2,tuning,7,7000,0.270775,-0.056707,0.056707,0.327482,0.330045
3,test,14,14000,0.268616,-0.063148,0.063148,0.331764,0.331370


Оценка test независима от финального переобучения и получена до него. Даты ранее изучались.


,выборка,даты,строки,WAPE,Relative Bias,|Relative Bias|,метрика конкурса,среднее по дням
0,train,67,67000,0.175008,-0.000045,0.000045,0.175053,0.187592
1,val,7,7000,0.276307,-0.038941,0.038941,0.315248,0.317125
2,tuning,7,7000,0.270775,-0.056707,0.056707,0.327482,0.330045
3,test,14,14000,0.268616,-0.063148,0.063148,0.331764,0.331370
4,вся история после refit,95,95000,0.190020,-0.000034,0.000034,0.190054,0.203959


... (35 outputs left out)

,выборка,даты,строки,WAPE,Relative Bias,|Relative Bias|,метрика конкурса,среднее по дням
0,test,14,112000,0.330651,-0.047926,0.047926,0.378578,0.378332
1,test profile28,14,112000,0.344260,0.014650,0.014650,0.358911,0.373950


,выборка,даты,строки,WAPE,Relative Bias,|Relative Bias|,метрика конкурса,среднее по дням
0,tuning,21,168000,0.331210,-0.031696,0.031696,0.362906,0.364343
1,tuning profile28,21,168000,0.344554,0.005623,0.005623,0.350176,0.365530


,window,model,profile_weight,выборка,даты,строки,WAPE,Relative Bias,|Relative Bias|,метрика конкурса,среднее по дням
0,1,XGBoost,0.00,train,7,56000,0.327748,-0.034422,0.034422,0.362170,0.362179
1,1,XGBoost + profile28,0.45,train,7,56000,0.326778,-0.017673,0.017673,0.344452,0.347476
2,1,XGBoost,0.00,val,7,56000,0.334451,-0.022379,0.022379,0.356830,0.361374
3,1,XGBoost + profile28,0.45,val,7,56000,0.333905,-0.009455,0.009455,0.343360,0.355634
4,2,XGBoost,0.00,train,14,112000,0.331071,-0.028452,0.028452,0.359523,0.361777
5,2,XGBoost + profile28,0.45,train,14,112000,0.330311,-0.013599,0.013599,0.343910,0.351555
6,2,XGBoost,0.00,val,7,56000,0.331502,-0.038473,0.038473,0.369975,0.369475
7,2,XGBoost + profile28,0.45,val,7,56000,0.330827,-0.017626,0.017626,0.348453,0.355599


,1,2,3,4,5,6,7,8,mean_all_8
status_1_mean_12,0.000381,0.070539,0.238704,0.165157,0.176462,0.091348,0.066394,0.061005,0.108749
target_1h_mean_12,0.000827,0.003664,0.000463,0.004370,0.020025,0.141922,0.212649,0.165359,0.068660
warehouse_total_load,0.376835,0.025964,0.001618,0.001282,0.001333,0.000869,0.005428,0.008687,0.052752
target_1h_mean_12_offset_m2,0.000356,0.000417,0.022190,0.004169,0.153842,0.220032,0.009634,0.000778,0.051427
target_1h_mean_12_offset_m4,0.004343,0.023296,0.140614,0.067412,0.029905,0.011875,0.001066,0.020086,0.037325
...,...,...,...,...,...,...,...,...,...
lagged_status6_signal_available_offset_p4,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
residual_lagged_status6_signal_available_offset_p3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
residual_lagged_status6_signal_available_offset_p2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
residual_lagged_status6_signal_available_offset_p4,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
